# Data PreProcessing

Importing and Loading

In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('dataset/movies_metadata.csv', dtype={'popularity': str})
credits = pd.read_csv('dataset/credits.csv')
keywords = pd.read_csv('dataset/keywords.csv')

Merging

In [3]:
movies['budget'] = pd.to_numeric(movies['budget'], errors='coerce').fillna(0).astype('int64')
movies['revenue'] = pd.to_numeric(movies['revenue'], errors='coerce').fillna(0).astype('int64')
movies['id'] = pd.to_numeric(movies['id'], errors='coerce').fillna(0).astype('int64')
movies['popularity'] = pd.to_numeric(movies['popularity'], errors='coerce').astype('float64')
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')


In [4]:
movies = movies.merge(credits, on='id').merge(keywords, on='id')
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 46628 entries, 0 to 46627
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  46628 non-null  str           
 1   belongs_to_collection  4574 non-null   str           
 2   budget                 46628 non-null  int64         
 3   genres                 46628 non-null  str           
 4   homepage               8009 non-null   str           
 5   id                     46628 non-null  int64         
 6   imdb_id                46611 non-null  str           
 7   original_language      46617 non-null  str           
 8   original_title         46628 non-null  str           
 9   overview               45633 non-null  str           
 10  popularity             46624 non-null  float64       
 11  poster_path            46229 non-null  str           
 12  production_companies   46624 non-null  str           
 13  production_c

Feature Selection

In [5]:
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

Cleaning Data

In [6]:
movies = movies.dropna(subset=['title', 'overview'])

In [7]:
movies.drop_duplicates(subset=['id'], keep='first', inplace=True)

In [8]:
required_cols = ['title', 'overview', 'genres', 'keywords', 'cast', 'crew']

def is_empty(value):
    if pd.isna(value):
        return True
    value = str(value).strip()
    return value == '' or value == '[]'

movies = movies.dropna(subset=required_cols)

for col in required_cols:
    movies = movies[~movies[col].apply(is_empty)]

In [9]:
movies.count()

id          29629
title       29629
overview    29629
genres      29629
keywords    29629
cast        29629
crew        29629
dtype: int64

Data Preprocessing

In [10]:
import ast

def convert(data):
    converted = []
    for i in ast.literal_eval(data):
        converted.append(i['name'])
    return converted

In [11]:
movies['genres'] = movies['genres'].apply(convert).apply(lambda x: [i.replace(" ", "") for i in x])

movies['keywords'] = movies['keywords'].apply(convert).apply(lambda x: [i.replace(" ", "") for i in x])

In [12]:
def convert_count(data):
    c = 0
    converted = []
    for i in ast.literal_eval(data):
        if(c < 5):
            converted.append(i['name'])
            c += 1
    return converted

In [13]:
movies['cast'] = movies['cast'].apply(convert_count).apply(lambda x: [i.replace(" ", "") for i in x])

In [14]:
import re

pattern = re.compile(r"'job':\s*'(Director|Director of Photography|Screenplay|Original Music Composer)'.*?'name':\s*'([^']+)'")

def extract_crew(data):
    crew_list = []
    for match in pattern.finditer(data):
        crew_list.append(match.group(2).replace(" ", ""))
    return crew_list

In [15]:
movies['crew'] = movies['crew'].apply(extract_crew)

In [16]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [17]:
movies['metadata'] = movies['overview'] + movies['genres'] + movies['keywords']
movies['ppl'] = movies['cast'] + movies['crew']

new_movies = movies[['id', 'title', 'metadata', 'ppl']]
new_movies['metadata'] = new_movies['metadata'].apply(lambda x: " ".join(x)).apply(lambda x: x.lower())
new_movies['ppl'] = new_movies['ppl'].apply(lambda x: " ".join(x)).apply(lambda x: x.lower())

In [18]:
def rm_punctuations(text):
    import string
    return text.translate(str.maketrans('', '', string.punctuation))

In [19]:
new_movies['metadata'] = new_movies['metadata'].apply(rm_punctuations)

# Vectorization

Stemming & Lemmatization

In [20]:
import nltk
from nltk.stem.snowball import SnowballStemmer

sb = SnowballStemmer("english")

In [21]:
def stem(text):
    L = []
    for i in text.split():
        L.append(sb.stem(i))
    return " ".join(L)

In [22]:
new_movies['metadata'] = new_movies['metadata'].apply(stem)

In [23]:
new_movies['tags'] = new_movies['metadata'] + new_movies['ppl']

Vectorization

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

counter = CountVectorizer(max_features=50000)
vecs = counter.fit_transform(new_movies['tags']).toarray()

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(vecs)

# Recommendation

In [26]:
def recommend(movie):
    movie_index = new_movies[new_movies['title'] == movie].index[0]
    distances = sim[movie_index]

    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:7]

    for i in movies_list:
        print(new_movies.iloc[i[0]].title)

In [27]:
recommend("Jumanji")

Indie Game: The Movie
Stay Alive
Godzilla vs. Mothra
Beta Test
Make-Out with Violence
Video Games: The Movie


Saving

In [28]:
import joblib
joblib.dump({"vectorizer": counter, "similarity": sim, "movies": new_movies}, "movie_recommender.joblib")

['movie_recommender.joblib']

# How to load

import joblib and create these variables and then create the recommend function and then use recommend("your_fav movie) :) <br>
    <pre>data = joblib.load("movie_recommender.joblib")<br>
    <pre>count = data.get("vectorizer")<br>
    <pre>sim = data["similarity"]<br>
    <pre>new_movies = data["movies"]<br>